# Part 1 — Build a CNN detector

We will now build a small ResNet-style CNN for detection.

In this notebook you will:

1. build a ResNet9 detector,
2. define the detection loss, and
3. implement the training and evaluation loops.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader, Subset

import assignment2 as a2

DATASET_ROOT = "./data/VMRD_COCO"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Build the ResNet9 detector

**Your task here:**

Complete the `ResNet9Detector` class in [`assignment2/model.py`](assignment2/model.py). (2 marks)
- `__init__()` (1 mark)
- `forward()` (1 mark)

The model architecture is summarized in the image below.

![](assets/model_architecture.png)

The output shape is `(B, 15, 16, 16)` because every grid cell predicts one
objectness logit, 10 class logits, and four box values.

Useful documentation:

- [Building NN](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [nn.Conv2d](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Conv2d.html#torch.nn.Conv2d)
- [nn.BatchNorm2d](https://docs.pytorch.org/docs/2.14/generated/torch.nn.BatchNorm2d.html)
- [nn.MaxPool2d](https://docs.pytorch.org/docs/2.14/generated/torch.nn.MaxPool2d.html#torch.nn.MaxPool2d)
- [nn.ReLU](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html#torch.nn.ReLU)

You can use the following code to check the output shape of your model:


In [ ]:
# Build a small DataLoader for a quick sanity check.
train_loader = a2.get_train_loader(DATASET_ROOT, batch_size=4)
toy_dataset = Subset(train_loader.dataset, range(8))
toy_loader = DataLoader(toy_dataset, batch_size=2, shuffle=True)

# Create the model.
model = a2.ResNet9Detector().to(device)

# Test one forward pass.
images, objectness_targets, class_targets, box_targets = next(iter(toy_loader))
images = images.to(device)
predictions = model(images)

print("Images:      ", images.shape)
print("Predictions:", predictions.shape)
print("Expected:   ", (len(images), 15, a2.GRID_SIZE, a2.GRID_SIZE))


## 2. Loss function

The total detection loss is

$$L = L_{obj} + L_{cls} + \lambda L_{box}.$$

We provide the objectness and box losses in
[`assignment2/training.py`](assignment2/training.py). You do not need to
modify these helper functions:

- **Objectness loss** (`calculate_objectness_loss()`): focal binary cross
  entropy over all grid cells. It gives less weight to easy cells so that
  the many easy background cells do not dominate the loss.
- **Box loss** (`calculate_box_loss()`): Smooth L1 loss plus generalized IoU
  (GIoU) loss, calculated only for cells containing objects. IoU measures
  the overlap between two boxes; a larger value means better overlap.
- `select_object_cells()` keeps only the cells containing objects. It turns a
  tensor of shape `(B, K, H, W)` into shape `(N, K)`, where `N` is the number
  of object cells.

The class loss is mean cross entropy over object cells only. The weight
$\lambda=2$ controls the contribution of the box loss.

**Your task here:**

1. (1 mark) Complete `logits_to_pred()` in
   [`assignment2/model.py`](assignment2/model.py) to split the model output
   into:
   - `objectness_logits`: the first channel;
   - `class_logits`: the next 10 channels;
   - `pred_boxes`: the last four channels, with sigmoid applied so the values
     are in `[0, 1]`.
2. (1 mark) Complete `detection_loss()` in
   [`assignment2/training.py`](assignment2/training.py):
   - use `select_object_cells()` and `object_cells` to select the class
     predictions and targets of the object cells, then use cross entropy to
     calculate `class_loss`;
   - combine the three losses into `total_loss` using the equation above.

Useful documentation:

- [Cross Entropy Loss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.cross_entropy.html)


In [ ]:
gt_objectness = objectness_targets.to(device)
gt_class = class_targets.to(device)
gt_boxes = box_targets.to(device)
objectness_logits, class_logits, pred_boxes = a2.logits_to_pred(predictions)

total_loss, objectness_loss, class_loss, box_loss = a2.detection_loss(
    predictions, gt_objectness, gt_class, gt_boxes
)

print("Objectness logits:", objectness_logits.shape)
print("Class logits:     ", class_logits.shape)
print("Box predictions:  ", pred_boxes.shape)
print("Total loss:     ", total_loss.item())
print("Objectness loss:", objectness_loss.item())
print("Class loss:     ", class_loss.item())
print("Box loss:       ", box_loss.item())

## 3. Training loop

**Your task here:**

(2 marks) Complete `train_detection_model()` in
[`assignment2/training.py`](assignment2/training.py). The `loss_history`
dictionary has been created for you. Your implementation should:

1. Create an AdamW optimizer with learning rate `2.5e-3` and weight decay
   `1e-4`. Create a OneCycleLR scheduler with `max_lr=2.5e-3`,
   `epochs=num_epochs`, `steps_per_epoch=len(train_loader)`, and
   `pct_start=0.15`. (0.5 mark)
2. For every mini-batch, run the forward pass, calculate the losses, run
   backpropagation, update the model, update the scheduler, and record each
   loss in `loss_history`. (1.5 marks)

Notes:

1. Call `model.train()` before training. Layers such as Batch Normalization
   behave differently during training and evaluation, and the model may have
   been switched to evaluation mode by an earlier call to
   `evaluate_detection_model()`.
2. Move the images and targets to `device` before passing them to the model.
3. Call `optimizer.zero_grad()`, `optimizer.step()`, and `scheduler.step()` in
   the appropriate order. OneCycleLR is updated after every mini-batch.
4. You may write the optimizer and scheduler arguments directly or use
   `**kwargs` to make them configurable.

Useful documentation:

- [Optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [OneCycleLR](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.OneCycleLR.html)


In [ ]:
loss_history = a2.train_detection_model(
    model,
    toy_loader,
    num_epochs=2,
    device=device,
)

for name, values in loss_history.items():
    print(f"{name}: {values[0]:.4f} -> {values[-1]:.4f}")

## 4. Evaluation loop

A prediction counts as a true positive when it has the correct class and its
IoU with an unmatched ground-truth box is at least 0.5. Precision is the
fraction of predictions that are correct, while recall is the fraction of
ground-truth objects that are detected. Decoding, matching, and NMS are
provided in `assignment2/evaluation.py`.

**Your task here:**

(2 marks) Complete `evaluate_detection_model()` in
[`assignment2/training.py`](assignment2/training.py).

Notes:

1. Use `model.eval()` and `with torch.no_grad()` properly during evaluation.
2. Move each mini-batch to the selected device before the forward pass. After
   calculating the logits of a mini-batch, move them and the targets back to
   the CPU right away. Keeping the results of every mini-batch on the GPU can
   run out of GPU memory.
3. The values in `average_losses` should be Python floats averaged over the
   complete dataset. Think carefully about how mini-batch losses should be
   combined.
4. We implemented `calculate_detection_accuracy()` for you, and the code after
   your implementation already calls it. Before that call, you must define the
   following four NumPy arrays, each containing the results of the complete
   dataset:
   - `predicted_logits` with shape `(N, 15, 16, 16)`;
   - `all_gt_objectness` with shape `(N, 16, 16)`;
   - `all_gt_class` with shape `(N, 16, 16)`;
   - `all_gt_boxes` with shape `(N, 4, 16, 16)`.

   It decodes predictions, applies NMS, and returns a dictionary containing
   precision and recall. The small DataLoader below is a sanity check.


In [ ]:
average_losses, detection_accuracy = a2.evaluate_detection_model(
    model, toy_loader, device
)

for name, value in average_losses.items():
    print(f"{name}: {value:.4f}")
for name, value in detection_accuracy.items():
    print(f"{name}: {value:.3f}")